## Final Model Evaluation 

In [4]:
import sys
import json
import joblib
import warnings
import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import matplotlib.cm as cm
from pathlib import Path
from sklearn.metrics import (
    roc_auc_score, average_precision_score,
    roc_curve, precision_recall_curve,
    confusion_matrix, f1_score
)
from sklearn.calibration import calibration_curve
from lifelines import KaplanMeierFitter
import scipy.integrate
if not hasattr(scipy.integrate, 'simps'):
    scipy.integrate.simps = scipy.integrate.simpson

warnings.filterwarnings('ignore')
matplotlib.rcParams.update({
    'figure.dpi'       : 150,
    'savefig.dpi'      : 300,
    'font.family'      : 'DejaVu Sans',
    'axes.spines.top'  : False,
    'axes.spines.right': False,
    'axes.grid'        : True,
    'grid.alpha'       : 0.3,
    'grid.linewidth'   : 0.5,
})

OUTPUT_DIR = Path('C:/Users/20220505/Desktop/output path')
FIG_DIR    = OUTPUT_DIR / 'figures_04'
FIG_DIR.mkdir(exist_ok=True)

HORIZONS  = [6, 12, 24]
NUM_BINS  = 48
MAX_HOURS = 200
time_cuts = np.linspace(0, MAX_HOURS, NUM_BINS + 1)[1:]

def savefig(name):
    plt.savefig(str(FIG_DIR / f'{name}.png'), dpi=300, bbox_inches='tight')
    plt.show()
    print(f'Saved -> {name}.png ✓')

print(f'OUTPUT_DIR : {OUTPUT_DIR}')
print(f'FIG_DIR    : {FIG_DIR}')
print('Setup complete ✓')

OUTPUT_DIR : C:\Users\20220505\Desktop\output path
FIG_DIR    : C:\Users\20220505\Desktop\output path\figures_04
Setup complete ✓


In [5]:
print('Loading survival model outputs...')

surv_ddh_inc = np.load(str(OUTPUT_DIR / 'ddh_incident_surv_calibrated.npy'))
surv_ddh_all = np.load(str(OUTPUT_DIR / 'ddh_all_surv_cal.npy'))
dur_ddh_inc  = np.load(str(OUTPUT_DIR / 'transformer_incident_dur.npy'))
evt_ddh_inc  = np.load(str(OUTPUT_DIR / 'transformer_incident_evt.npy'))
dur_ddh_all  = np.load(str(OUTPUT_DIR / 'ddh_all_dur.npy'))
evt_ddh_all  = np.load(str(OUTPUT_DIR / 'ddh_all_evt.npy'))

surv_t_inc = np.load(str(OUTPUT_DIR / 'transformer_incident_surv_cal.npy'))
surv_t_all = np.load(str(OUTPUT_DIR / 'transformer_all_surv_cal.npy'))
dur_t_inc  = np.load(str(OUTPUT_DIR / 'transformer_incident_dur.npy'))
evt_t_inc  = np.load(str(OUTPUT_DIR / 'transformer_incident_evt.npy'))
dur_t_all  = np.load(str(OUTPUT_DIR / 'transformer_all_dur.npy'))
evt_t_all  = np.load(str(OUTPUT_DIR / 'transformer_all_evt.npy'))

shap_values = np.load(str(OUTPUT_DIR / 'shap_values_ddh.npy'))
shap_times  = np.load(str(OUTPUT_DIR / 'shap_times.npy'))

with open(OUTPUT_DIR / 'feature_names.txt') as f:
    feature_cols = f.read().splitlines()
with open(OUTPUT_DIR / 'rich_feature_names.txt') as f:
    rich_feature_names = f.read().splitlines()
with open(OUTPUT_DIR / 'results_summary.json') as f:
    results_summary = json.load(f)

horizon_results = joblib.load(str(OUTPUT_DIR / 'horizon_results_full.pkl'))
survival_df     = pd.read_csv(str(OUTPUT_DIR / 'survival_dataset.csv'))
survival_inc    = pd.read_csv(str(OUTPUT_DIR / 'survival_incident_dataset.csv'))

print(f'  surv_ddh_inc : {surv_ddh_inc.shape}')
print(f'  surv_t_inc   : {surv_t_inc.shape}')
print(f'  surv_t_all   : {surv_t_all.shape}')
print(f'  shap_values  : {shap_values.shape}')
print(f'  feature_cols : {len(feature_cols)}')

print('\nLoading rolling window outputs...')
rolling_df = pd.read_csv(str(OUTPUT_DIR / 'rolling_predictions.csv'))
with open(OUTPUT_DIR / 'rolling_results.json') as f:
    rolling_results = json.load(f)

hl = pd.read_csv(str(OUTPUT_DIR / 'hourly_labels.csv'))
onset_lookup = (
    hl[hl['label'] == 1]
    .groupby('stay_id')['hour'].min()
    .to_dict()
)

print(f'  rolling_df   : {rolling_df.shape}')
print(f'  onset_lookup : {len(onset_lookup):,} sepsis stays')
print('\nAll outputs loaded ✓')

Loading survival model outputs...
  surv_ddh_inc : (7310, 48)
  surv_t_inc   : (7310, 48)
  surv_t_all   : (10604, 48)
  shap_values  : (7887, 127, 12)
  feature_cols : 127

Loading rolling window outputs...
  rolling_df   : (2296449, 6)
  onset_lookup : 26,643 sepsis stays

All outputs loaded ✓


In [ ]:
from pycox.evaluation import EvalSurv

print('Computing bootstrap CIs (1000 resamples)...')
print('This takes ~2-3 minutes...')

N_BOOTSTRAP = 1000
RNG         = np.random.default_rng(42)

def surv_to_df(surv, cuts):
    return pd.DataFrame(surv.T, index=cuts)

def compute_ci(surv, dur, evt, cuts):
    ev = EvalSurv(surv_to_df(surv, cuts), dur, evt, censor_surv='km')
    return ev.concordance_td()

def compute_ibs(surv, dur, evt, cuts):
    ev          = EvalSurv(surv_to_df(surv, cuts), dur, evt, censor_surv='km')
    event_times = dur[evt == 1]
    t_min       = np.percentile(event_times, 5)
    t_max       = np.percentile(event_times, 95)
    return ev.integrated_brier_score(np.linspace(t_min, t_max, 100))

def bootstrap_metric(fn, surv, dur, evt, cuts, n=N_BOOTSTRAP):
    scores    = []
    n_samples = len(dur)
    for _ in range(n):
        idx = RNG.integers(0, n_samples, size=n_samples)
        if evt[idx].sum() == 0:
            continue
        try:
            scores.append(fn(surv[idx], dur[idx], evt[idx], cuts))
        except Exception:
            continue
    scores = np.array(scores)
    return scores.mean(), np.percentile(scores, 2.5), np.percentile(scores, 97.5)

models = [
    ('DDH (cal)',         'Incident',   surv_ddh_inc, dur_ddh_inc, evt_ddh_inc),
    ('DDH (cal)',         'All Sepsis', surv_ddh_all, dur_ddh_all, evt_ddh_all),
    ('Transformer (cal)', 'Incident',   surv_t_inc,   dur_t_inc,   evt_t_inc),
    ('Transformer (cal)', 'All Sepsis', surv_t_all,   dur_t_all,   evt_t_all),
]

rows = []
for model_name, dataset, surv, dur, evt in models:
    print(f'\n  {model_name} | {dataset}')
    ci_pt  = compute_ci(surv, dur, evt, time_cuts)
    ibs_pt = compute_ibs(surv, dur, evt, time_cuts)
    ci_mu,  ci_lo,  ci_hi  = bootstrap_metric(compute_ci,  surv, dur, evt, time_cuts)
    ibs_mu, ibs_lo, ibs_hi = bootstrap_metric(compute_ibs, surv, dur, evt, time_cuts)
    rows.append({
        'Model'      : model_name,
        'Dataset'    : dataset,
        'C-index'    : round(ci_pt,  4),
        'CI_lo'      : round(ci_lo,  4),
        'CI_hi'      : round(ci_hi,  4),
        'IBS'        : round(ibs_pt, 4),
        'IBS_lo'     : round(ibs_lo, 4),
        'IBS_hi'     : round(ibs_hi, 4),
        'N_test'     : len(dur),
        'N_events'   : int(evt.sum()),
        'Event_rate' : round(evt.mean(), 4),
    })
    print(f'    C-index : {ci_pt:.4f}  95% CI [{ci_lo:.4f}, {ci_hi:.4f}]')
    print(f'    IBS     : {ibs_pt:.4f}  95% CI [{ibs_lo:.4f}, {ibs_hi:.4f}]')

bootstrap_df = pd.DataFrame(rows)
bootstrap_df.to_csv(str(OUTPUT_DIR / 'bootstrap_results.csv'), index=False)
print('\nSaved -> bootstrap_results.csv ✓')

Computing bootstrap CIs (1000 resamples)...
This takes ~2-3 minutes...

  DDH (cal) | Incident
